In [1]:
import sys
import os
import nltk

# Dodanie lokalnej ścieżki NLTK
nltk.data.path.append(os.path.abspath('../data/nltk_data'))

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from src import text_prep

import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# 1. Ewentualne pobranie zasobów (tylko za pierwszym razem)
# text_prep.download_nltk_resources()

# 2. Pobranie danych 20 Newsgroups
# Warto użyć parametru 'remove', aby usunąć metadane, które psują klastrowanie (model uczyłby się np. adresów email zamiast tematyki)
print("Pobieranie danych...")
dataset = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
documents = dataset.data

# Zmniejszamy zbiór na czas testów, żeby szybciej się liczyło (opcjonalnie)
documents_subset = documents[:1000]

# 3. Preprocessing (To potrwa chwilę!)
print("Rozpoczęcie preprocessingu...")
processed_docs = text_prep.preprocess_corpus(documents_subset)
print("Preprocessing zakończony.")

# Podgląd przed i po:
print("\n--- PRZED ---")
print(documents_subset[0][:200])
print("\n--- PO ---")
print(processed_docs[0][:200])

# 4. Reprezentacja TF-IDF
vectorizer = TfidfVectorizer(
    max_df=0.95,        # ignoruj słowa występujące w więcej niż 95% dokumentów
    min_df=5,           # ignoruj słowa występujące w mniej niż 5 dokumentach
    max_features=5000   # ogranicz rozmiar słownika do 5000 najważniejszych słów
)

tfidf_matrix = vectorizer.fit_transform(processed_docs)
print(f"\nKształt macierzy TF-IDF: {tfidf_matrix.shape}")

# 5. Zapisanie przetworzonych danych do późniejszego użytku
os.makedirs('../data', exist_ok=True)
data_to_save = {
    'documents': documents_subset,
    'processed_docs': processed_docs,
    'tfidf_matrix': tfidf_matrix,
    'vectorizer': vectorizer,
    'labels': dataset.target[:1000],
    'target_names': dataset.target_names
}
joblib.dump(data_to_save, '../data/tfidf_data.joblib')
print("\nDane zostały zapisane do pliku: data/tfidf_data.joblib")

Pobieranie danych...
Rozpoczęcie preprocessingu...
Preprocessing zakończony.

--- PRZED ---


I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However,

--- PO ---
sure bashers pen fan pretty confused lack kind post recent pen massacre devil actually bit puzzled bit relieved however going put end non pittsburghers relief bit praise pen man killing devil worse th

Kształt macierzy TF-IDF: (1000, 2434)

Dane zostały zapisane do pliku: data/tfidf_data.joblib
